# Diagnóstico de qualidade dos dados

Este notebook reúne a investigação da qualidade das tabelas Bronze e decisões para a criação da tabela prata

Para cada tabela são verificadas as dimensões de qualidade pedidas no enunciado:

- **Completude:** valores nulos ou vazios.
- **Consistência:** valores seguem o padrão esperado (formato de ID, grafia de cidade, UF, ordem das datas).
- **Unicidade:** a chave da tabela não se repete.
- **Acurácia:** os valores fazem sentido no contexto (datas plausíveis, preços compatíveis com o produto).
- **Outliers:** valores extremos que possam distorcer as análises.

Também são testadas as relações entre tabelas (integridade referencial), que sustentam os joins da camada ouro.

## 1. customers

Granularidade esperada: uma linha por `customer_id`. No Olist, o `customer_id` é gerado a cada pedido quem identifica a pessoa é o `customer_unique_id`. Essa distinção é verificada abaixo porque muda qualquer análise de recompra.

### 1.1 Estados

Lista cada UF com a quantidade de clientes. Como o domínio é fechado e pequeno (27 UFs), basta comparar os valores com a lista oficial. Um valor inválido, em minúsculas ou nulo apareceria como uma linha a mais.

**Resultado:** 27 valores, todos UFs válidas em maiúsculas, sem nulos. Nenhuma correção necessária.

In [0]:
SELECT DISTINCT customer_state
FROM pospucrio.bronze.olist_customers_dataset
GROUP BY customer_state
ORDER BY customer_state

### 1.2 Cidades: padrão de escrita

As cidades têm milhares de valores distintos, o que impede a verificação visual. Esta consulta confirma o padrão aparente (minúsculas, sem acento) listando toda cidade com algum caractere fora de letras minúsculas e espaço.

**Resultado:** 53 cidades, todas com apóstrofo ou hífen em nomes legítimos ("olho d'agua das flores", "ceara-mirim"). Nenhum acento, maiúscula ou número. O padrão está confirmado.

In [0]:
SELECT customer_city, COUNT(*) AS n
FROM pospucrio.bronze.olist_customers_dataset
WHERE customer_city RLIKE '[^a-z ]'
GROUP BY customer_city
ORDER BY n

### 1.3 Cidades: variantes de grafia

Cria uma chave normalizada removendo tudo que não é letra e agrupa por estado e chave. Se duas grafias diferentes colapsam na mesma chave, são a mesma cidade escrita de formas distintas. O estado entra no agrupamento para não confundir cidades homônimas de UFs diferentes.

**Resultado:** 6 cidades com duas grafias. Cinco são o apóstrofo substituído por espaço ("dias d avila" e "dias d'avila", "estrela d oeste" e "estrela d'oeste"), padrão resolvido por regra na função `padronizar_cidade`. Uma é o hífen ("mogi-mirim" e "mogi mirim"), resolvida como correção pontual.


In [0]:
SELECT customer_state,
       regexp_replace(customer_city, '[^a-z]', '') AS chave,
       collect_set(customer_city) AS variantes
FROM pospucrio.bronze.olist_customers_dataset
GROUP BY customer_state, chave
HAVING size(collect_set(customer_city)) > 1

### 1.4 Cidades: possíveis erros de digitação

A chave normalizada não detecta erros de digitação, esta consulta compara as cidades de um mesmo estado pela distância de Levenshtein (quantas letras precisam mudar para uma palavra virar a outra) e lista os pares com distância até 2.

**Resultado:** 415 pares candidatos. Além dos casos de apóstrofo já identificados em 1.3, a inspeção mostra municípios reais com nomes parecidos (Crisópolis e Cristópolis, Teofilândia e Teolândia, São Felipe e São Félix, na Bahia). Não foi encontrado erros de digitação.

In [0]:
WITH cidades AS (
  SELECT DISTINCT customer_state AS uf, customer_city AS cidade
  FROM pospucrio.bronze.olist_customers_dataset
)
SELECT a.uf, a.cidade, b.cidade AS parecida, levenshtein(a.cidade, b.cidade) AS dist
FROM cidades a
JOIN cidades b
  ON a.uf = b.uf AND a.cidade < b.cidade
WHERE levenshtein(a.cidade, b.cidade) <= 2
ORDER BY a.uf, dist

### 1.5 IDs e CEP

Foram verificadas se os id cumprem sua função: existir, seguir um formato consistente e ser único onde deveria ser. O CEP deve ter 5 dígitos, menos que isso indicaria erro.

**Resultado:** 99.441 linhas e 99.441 `customer_id` distintos, confirmando a chave. Nenhum ID vazio ou fora do padrão. 96.096 pessoas distintas pelo `customer_unique_id`. Todos os CEPs com 5 dígitos.

In [0]:
SELECT
  COUNT(*)                                                AS linhas,
  COUNT_IF(customer_id IS NULL OR customer_id = '')       AS id_vazio,
  COUNT_IF(customer_id NOT RLIKE '^[0-9a-f]{32}$')        AS id_fora_padrao,
  COUNT_IF(customer_unique_id NOT RLIKE '^[0-9a-f]{32}$') AS unique_fora_padrao,
  COUNT(DISTINCT customer_id)                             AS ids_distintos,
  COUNT(DISTINCT customer_unique_id)                      AS pessoas_distintas,
  COUNT_IF(length(customer_zip_code_prefix) <> 5)         AS cep_tamanho_estranho
FROM pospucrio.bronze.olist_customers_dataset

### 1.6 Relação entre customer_id e customer_unique_id

Mostra quantas pessoas têm 1, 2, 3 ou mais `customer_id`. O esperado é a grande maioria com 1 e uma cauda pequena de recompradores. Uma pessoa com centenas de IDs indicaria um valor genérico usado como identificador.

**Resultado:** 93.099 pessoas (96,9%) compraram uma única vez, 2.745 duas vezes, 203 três vezes, o máximo é uma pessoa com 17 pedidos. Distribuição plausível. Confirma que contar `customer_id` como cliente superestimaria o número de pessoas e esconderia as recompras.

In [0]:
SELECT n_ids, COUNT(*) AS pessoas
FROM (
  SELECT customer_unique_id, COUNT(*) AS n_ids
  FROM pospucrio.bronze.olist_customers_dataset
  GROUP BY customer_unique_id
)
GROUP BY n_ids
ORDER BY n_ids

### 1.7 Integridade com pedidos

Todo pedido deve ter um cliente, e todo cliente desta tabela deve ter um pedido. Se algum valor fosse diferente de zero, joins entre pedidos e clientes perderiam registros sem aviso.

**Resultado:** 0 e 0. Integridade completa.

In [0]:
SELECT
  (SELECT COUNT(*) FROM pospucrio.bronze.olist_orders_dataset o
     LEFT ANTI JOIN pospucrio.bronze.olist_customers_dataset c
     ON o.customer_id = c.customer_id) AS pedidos_sem_cliente,
  (SELECT COUNT(*) FROM pospucrio.bronze.olist_customers_dataset c
     LEFT ANTI JOIN pospucrio.bronze.olist_orders_dataset o
     ON c.customer_id = o.customer_id) AS clientes_sem_pedido

### 1.8 Validação da prata

Repete a busca de variantes da seção 1.3 sobre a tabela limpa. Deve retornar vazio, comprovando que as regras de padronização resolveram os casos encontrados.

**Resultado:** vazio.

In [0]:
SELECT customer_state,
       regexp_replace(customer_city, '[^a-z]', '') AS chave,
       collect_set(customer_city) AS variantes
FROM pospucrio.silver.customers
GROUP BY customer_state, chave
HAVING size(collect_set(customer_city)) > 1

## 2. sellers

Granularidade esperada: uma linha por `seller_id`. A estrutura é a mesma de clientes (ID, CEP, cidade, UF), mas a cidade é digitada pelo próprio vendedor no cadastro e tende a vir menos padronizada.

### 2.1 Granularidade, IDs e CEP

**Resultado:** 3.095 linhas e 3.095 vendedores distintos. Nenhum ID fora do padrão, nenhum CEP com tamanho diferente de 5, nenhuma cidade vazia. 23 UFs distintas.

In [0]:
SELECT
  COUNT(*)                                                AS linhas,
  COUNT(DISTINCT seller_id)                               AS vendedores_distintos,
  COUNT_IF(seller_id NOT RLIKE '^[0-9a-f]{32}$')          AS id_fora_padrao,
  COUNT_IF(length(trim(seller_zip_code_prefix)) <> 5)     AS cep_tamanho_estranho,
  COUNT(DISTINCT upper(trim(seller_state)))               AS ufs,
  COUNT_IF(seller_city IS NULL OR trim(seller_city) = '') AS cidade_vazia
FROM pospucrio.bronze.olist_sellers_dataset

### 2.2 Estados

Assim como em clientes se espera as 27 UFs no mesmo padrão. A verificação é se todos os valores são UFs válidas.

**Resultado:** 23 UFs, todas válidas. Forte concentração em SP (1.849 vendedores, 60%), seguido de PR (349) e MG (244).

In [0]:
SELECT seller_state, COUNT(*) AS vendedores
FROM pospucrio.bronze.olist_sellers_dataset
GROUP BY seller_state
ORDER BY seller_state

### 2.3 Cidades: caracteres fora do padrão

Lista as cidades que, mesmo após a função de padronização, mantêm caracteres fora de letras, espaço, apóstrofo e hífen. Mostra o que a regra geral não resolve.

**Resultado (obtido com a primeira versão da função):** 23 casos, agrupados em padrões:
- Estado digitado junto com a cidade, separado por barra, barra invertida, vírgula ou traço: "jacarei / sao paulo", "rio de janeiro, rio de janeiro, brasil", "pinhais/pr".
- Complemento entre parênteses: "arraial d'ajuda (porto seguro)".
- Acento agudo usado como apóstrofo: "santa barbara d´oeste".
- "são paulo" com acento remanescente, gravado com o til como caractere separado da letra.
- Valores sem cidade: "04482255" (número), "vendas@creditparts.com.br" (e-mail), "sp / sp" (apenas a sigla).

**Decisão:** os quatro primeiros padrões foram incorporados como regras na função `padronizar_cidade`. Os valores sem cidade entraram na lista de correções pontuais como nulos. Ao reexecutar esta consulta com a versão atual da função, os casos resolvidos deixam de aparecer, o estado final é verificado em 2.7.

In [0]:
SELECT seller_state,
       pospucrio.silver.padronizar_cidade(seller_city) AS cidade,
       COUNT(*) AS vendedores
FROM pospucrio.bronze.olist_sellers_dataset
WHERE pospucrio.silver.padronizar_cidade(seller_city) RLIKE "[^a-z '-]"
GROUP BY 1, 2
ORDER BY vendedores

### 2.4 Cidades: variantes de grafia

Mesma técnica da seção 1.3, aplicada após a função de padronização.

**Resultado (obtido com a primeira versão da função):** 5 grupos. Três foram resolvidos pelas regras novas da função ("são paulo" com acento, "d´oeste", barra invertida). Dois viraram correções pontuais: "portoferreira" (espaço faltando) e "sao paulo sp" (estado sem separador).

In [0]:
WITH p AS (
  SELECT seller_state AS uf,
         pospucrio.silver.padronizar_cidade(seller_city) AS cidade
  FROM pospucrio.bronze.olist_sellers_dataset
)
SELECT uf, regexp_replace(cidade, '[^a-z]', '') AS chave, collect_set(cidade) AS variantes
FROM p
GROUP BY uf, chave
HAVING size(collect_set(cidade)) > 1

### 2.5 Vendedores sem venda

Não seria erro, mas indica o tamanho da base ativa. O caminho inverso (item sem vendedor) está na seção 4.7.

**Resultado:** 0. Todo vendedor cadastrado tem ao menos uma venda.

In [0]:
SELECT COUNT(*) AS vendedores_sem_item
FROM pospucrio.bronze.olist_sellers_dataset s
LEFT ANTI JOIN pospucrio.silver.order_items i USING (seller_id)

### 2.6 Cidades sem valor recuperável: consulta pelo CEP

Para os três vendedores cuja cidade não é um nome de cidade, busca as cidades associadas ao prefixo de CEP na tabela de geolocalização do bronze. A tabela não segue para a prata, mas pode ser usada como referência.

**Resultado:** cada prefixo corresponde a uma única cidade, no mesmo estado informado pelo vendedor: 22790 é Rio de Janeiro (RJ), 03363 é São Paulo (SP) e 87025 é Maringá (PR). Os estados estão corretos, o problema é só o campo de cidade. O número "04482255" não é o CEP do vendedor, apenas um valor digitado no campo errado.

**Decisão:** as três cidades foram anuladas na prata, sem inferir o valor a partir de outra tabela, porque a cidade não é usada no modelo da camada ouro.

In [0]:
SELECT s.seller_state, s.seller_city, s.seller_zip_code_prefix,
       collect_set(pospucrio.silver.padronizar_cidade(g.geolocation_city)) AS cidades_do_cep,
       collect_set(g.geolocation_state)                                    AS ufs_do_cep
FROM pospucrio.bronze.olist_sellers_dataset s
LEFT JOIN pospucrio.bronze.olist_geolocation_dataset g
  ON lpad(trim(s.seller_zip_code_prefix), 5, '0') = lpad(trim(g.geolocation_zip_code_prefix), 5, '0')
WHERE s.seller_city IN ('04482255', 'vendas@creditparts.com.br', 'sp / sp')
GROUP BY 1, 2, 3

### 2.7 Validação da prata

Repete as consultas 2.3 e 2.4 sobre a tabela limpa. A primeira deve retornar apenas nomes legítimos com apóstrofo ou hífen, a segunda deve retornar vazio.

In [0]:
SELECT seller_state, seller_city, COUNT(*) AS vendedores
FROM pospucrio.silver.sellers
WHERE seller_city RLIKE "[^a-z '-]"
GROUP BY 1, 2
ORDER BY vendedores

In [0]:
SELECT seller_state,
       regexp_replace(seller_city, '[^a-z]', '') AS chave,
       collect_set(seller_city) AS variantes
FROM pospucrio.silver.sellers
GROUP BY seller_state, chave
HAVING size(collect_set(seller_city)) > 1

## 3. orders

Tabela central do modelo: define o status, as datas e o período da análise. Granularidade esperada é de uma linha por pedido.

### 3.1 Granularidade e tipos

Separa nulo de inválido. Datas de aprovação, postagem e entrega podem ser nulas legitimamente (pedido que não chegou àquela etapa) o que não pode é um valor preenchido que não converte. Compra e estimativa de entrega, todo pedido deve ter.

**Resultado:** 99.441 pedidos distintos e 99.441 clientes distintos, confirmando a relação de um `customer_id` por pedido. Nenhuma data inválida, todo pedido tem data de compra e estimativa de entrega.

In [0]:
SELECT
  COUNT(*)                                                               AS linhas,
  COUNT(DISTINCT order_id)                                               AS pedidos_distintos,
  COUNT(DISTINCT customer_id)                                            AS clientes_distintos,
  COUNT_IF(try_cast(order_purchase_timestamp AS TIMESTAMP) IS NULL)      AS compra_nula,
  COUNT_IF(order_approved_at IS NOT NULL
       AND try_cast(order_approved_at AS TIMESTAMP) IS NULL)             AS aprovacao_invalida,
  COUNT_IF(order_delivered_carrier_date IS NOT NULL
       AND try_cast(order_delivered_carrier_date AS TIMESTAMP) IS NULL)  AS postagem_invalida,
  COUNT_IF(order_delivered_customer_date IS NOT NULL
       AND try_cast(order_delivered_customer_date AS TIMESTAMP) IS NULL) AS entrega_invalida,
  COUNT_IF(try_cast(order_estimated_delivery_date AS TIMESTAMP) IS NULL) AS estimativa_nula
FROM pospucrio.bronze.olist_orders_dataset

### 3.2 Status contra datas

O status precisa concordar com as datas preenchidas. Um pedido entregue sem data de entrega, ou cancelado com data de entrega, é contradição.

**Resultado:**
- `delivered` (96.478): 8 sem data de entrega, 2 sem postagem, 14 sem aprovação.
- `canceled` (625): 6 com data de entrega. Pode ser devolução, mas o dataset não permite confirmar.
- `shipped`, `unavailable`, `invoiced`, `processing`, `created`, `approved`: consistentes, todos sem data de entrega.

**Decisão:** contradições mantidas e documentadas. As perguntas do objetivo usam o status, não essas datas.

In [0]:
SELECT
  order_status,
  COUNT(*)                                     AS pedidos,
  COUNT_IF(order_approved_at IS NULL)          AS sem_aprovacao,
  COUNT_IF(order_delivered_carrier_date IS NULL)  AS sem_postagem,
  COUNT_IF(order_delivered_customer_date IS NULL) AS sem_entrega
FROM pospucrio.bronze.olist_orders_dataset
GROUP BY order_status
ORDER BY pedidos DESC

### 3.3 Ordem cronológica

As etapas de um pedido têm ordem: compra, aprovação, postagem, entrega. Cada contagem é uma inversão dessa ordem.

**Resultado:** nenhuma aprovação antes da compra e nenhuma entrega antes da compra, que seriam os erros graves. 1.359 postagens antes da aprovação, com provável explicação operacional (a aprovação do pagamento registrada por outro sistema, com defasagem). 23 entregas antes da postagem, fisicamente impossíveis, tratadas como erro de registro.

**Decisão:** mantidos e documentados. Afetariam uma análise de prazos, não as perguntas do objetivo.

In [0]:
WITH d AS (
  SELECT
    CAST(order_purchase_timestamp AS TIMESTAMP)       AS compra,
    CAST(order_approved_at AS TIMESTAMP)              AS aprovacao,
    CAST(order_delivered_carrier_date AS TIMESTAMP)   AS postagem,
    CAST(order_delivered_customer_date AS TIMESTAMP)  AS entrega
  FROM pospucrio.bronze.olist_orders_dataset
)
SELECT
  COUNT_IF(aprovacao < compra)   AS aprovado_antes_da_compra,
  COUNT_IF(postagem  < aprovacao) AS postado_antes_da_aprovacao,
  COUNT_IF(entrega   < postagem) AS entregue_antes_da_postagem,
  COUNT_IF(entrega   < compra)   AS entregue_antes_da_compra
FROM d

### 3.4 Prazo de entrega e atraso

Mede o comportamento normal do tempo entre compra e entrega e da diferença entre a entrega real e a estimada (negativo significa entrega antes do prometido, positivo, atraso). Serve para identificar outliers de tempo e dá contexto para a análise das notas, já que o atraso é a explicação mais provável para avaliações ruins.

In [0]:
SELECT
  percentile_approx(datediff(CAST(order_delivered_customer_date AS TIMESTAMP),
                             CAST(order_purchase_timestamp AS TIMESTAMP)),
                    array(0.01, 0.5, 0.99))                  AS dias_ate_entrega,
  MAX(datediff(CAST(order_delivered_customer_date AS TIMESTAMP),
               CAST(order_purchase_timestamp AS TIMESTAMP)))  AS max_dias,
  percentile_approx(datediff(CAST(order_delivered_customer_date AS TIMESTAMP),
                             CAST(order_estimated_delivery_date AS TIMESTAMP)),
                    array(0.01, 0.5, 0.99))                  AS dias_vs_estimativa
FROM pospucrio.bronze.olist_orders_dataset
WHERE order_status = 'delivered'

### 3.5 Volume mensal e pedidos parados

Duas verificações numa consulta. O volume mensal mostra onde os dados começam e terminam de fato, e define o recorte temporal. A coluna `em_andamento` conta pedidos em status intermediário em cada mês de compra.

**Resultado, recorte temporal:**
- 2016: 4 pedidos em setembro, 324 em outubro, nenhum em novembro e 1 em dezembro. Volume baixo e uma lacuna na série. **Excluído da análise.**
- Agosto de 2018 é o último mês completo (6.512 pedidos), setembro tem 16 e outubro, 4. O último trimestre completo é o segundo de 2018.
- **Série trimestral:** do 1º trimestre de 2017 ao 2º de 2018 (seis trimestres).
- **Comparações entre anos:** janeiro a agosto em 2017 e em 2018. O primeiro semestre está completo nos dois anos.
- Novembro de 2017 (7.544 pedidos, contra 4.631 em outubro) registra o efeito da Black Friday.

**Resultado, pedidos parados:** os pedidos em status intermediário aparecem em volume estável (entre 37 e 165 por mês) desde janeiro de 2017. Se estivessem em curso, se concentrariam no fim do período. São pedidos que pararam e nunca foram concluídos.

**Decisão:** na camada ouro, apenas pedidos `delivered` contam como compra efetivada.

In [0]:
SELECT
  date_trunc('month', CAST(order_purchase_timestamp AS TIMESTAMP)) AS mes,
  COUNT(*)                                                         AS pedidos,
  COUNT_IF(order_status = 'delivered')                             AS entregues,
  COUNT_IF(order_status IN ('shipped', 'processing', 'invoiced',
                            'approved', 'created'))                AS em_andamento
FROM pospucrio.bronze.olist_orders_dataset
GROUP BY 1
ORDER BY 1

## 4. order_items

Granularidade esperada: uma linha por item de pedido, com chave composta `order_id` + `order_item_id`. Cada linha é uma unidade vendida: duas unidades do mesmo produto aparecem em duas linhas.

### 4.1 Chave, tipos e valores inválidos

**Resultado:** 112.650 linhas e 112.650 chaves distintas, granularidade confirmada. Nenhum preço, frete ou data inválido, nenhum preço zero ou negativo. 383 itens com frete zero, plausível como frete grátis. Prazo de postagem entre 2016 e 2020, a data máxima é implausível para um dataset que termina em 2018 e é investigada em 4.2.

In [0]:
SELECT
  COUNT(*)                                                          AS linhas,
  COUNT(DISTINCT order_id, order_item_id)                           AS chaves_distintas,
  COUNT_IF(try_cast(price AS DECIMAL(10,2)) IS NULL)                AS preco_invalido,
  COUNT_IF(try_cast(price AS DECIMAL(10,2)) <= 0)                   AS preco_zero_ou_negativo,
  COUNT_IF(try_cast(freight_value AS DECIMAL(10,2)) IS NULL)        AS frete_invalido,
  COUNT_IF(try_cast(freight_value AS DECIMAL(10,2)) < 0)            AS frete_negativo,
  COUNT_IF(try_cast(freight_value AS DECIMAL(10,2)) = 0)            AS frete_zero,
  COUNT_IF(try_cast(shipping_limit_date AS TIMESTAMP) IS NULL)      AS data_invalida,
  MIN(try_cast(shipping_limit_date AS TIMESTAMP))                   AS data_min,
  MAX(try_cast(shipping_limit_date AS TIMESTAMP))                   AS data_max
FROM pospucrio.bronze.olist_order_items_dataset

### 4.2 Prazos de postagem implausíveis

O `shipping_limit_date` é o prazo que o vendedor tem para entregar o produto à transportadora. A primeira consulta lista os itens com prazo após 2018 e a distância em dias até a compra. A segunda mede o normal, com os percentis 1, 50 e 99 dessa distância.

**Resultado:** 4 itens de pedidos de 2017 com prazo em 2020, a cerca de 1.050 dias da compra. A mediana normal é de 6 dias e o percentil 99, de 19 dias, os 4 casos estão mais de 50 vezes acima desse limite.

**Decisão:** erro de registro. Mantidos como vieram, pois a coluna não é usada no modelo da camada ouro.

In [0]:
SELECT
  i.order_id,
  o.order_status,
  CAST(o.order_purchase_timestamp AS TIMESTAMP)  AS data_compra,
  CAST(i.shipping_limit_date AS TIMESTAMP)       AS prazo_postagem,
  datediff(CAST(i.shipping_limit_date AS TIMESTAMP),
           CAST(o.order_purchase_timestamp AS TIMESTAMP)) AS dias
FROM pospucrio.bronze.olist_order_items_dataset i
JOIN pospucrio.bronze.olist_orders_dataset o USING (order_id)
WHERE CAST(i.shipping_limit_date AS TIMESTAMP) > '2018-12-31'
ORDER BY dias DESC

In [0]:
SELECT percentile_approx(
         datediff(CAST(i.shipping_limit_date AS TIMESTAMP),
                  CAST(o.order_purchase_timestamp AS TIMESTAMP)),
         array(0.01, 0.5, 0.99)) AS dias_percentis
FROM pospucrio.bronze.olist_order_items_dataset i
JOIN pospucrio.bronze.olist_orders_dataset o USING (order_id)

### 4.3 Outliers de preço e frete: distribuição

Os percentis descrevem o comportamento normal. O intervalo entre 0,25 e 0,75 é a metade central dos itens, acima do percentil 99 estão apenas 1% dos valores.

**Resultado:** preço com mediana de R$ 74,99, metade central entre R$ 39,90 e R$ 134,90, percentil 99 de R$ 889,99 e máximo de R$ 6.735,00. Frete com mediana de R$ 16,26, percentil 99 de R$ 84,57 e máximo de R$ 409,68. Distribuição típica de varejo, concentrada em valores baixos e com cauda longa.

**Consequência para a análise:** com essa cauda, a média é puxada para cima pelos poucos itens caros. Perguntas sobre valor médio devem reportar também a mediana.

In [0]:
SELECT
  percentile_approx(CAST(price AS DECIMAL(10,2)), array(0.01, 0.25, 0.5, 0.75, 0.99))         AS preco_percentis,
  MAX(CAST(price AS DECIMAL(10,2)))                                                             AS preco_max,
  percentile_approx(CAST(freight_value AS DECIMAL(10,2)), array(0.01, 0.25, 0.5, 0.75, 0.99)) AS frete_percentis,
  MAX(CAST(freight_value AS DECIMAL(10,2)))                                                     AS frete_max
FROM pospucrio.bronze.olist_order_items_dataset

### 4.4 Outliers de preço e frete: os extremos

Um valor distante não é necessariamente errado. Para julgar, os itens mais caros e os de frete mais alto são listados com categoria, peso e volume do produto.

**Resultado:** os itens mais caros pertencem a categorias compatíveis (computadores, eletroportáteis, consoles, instrumentos musicais, relógios). O frete acompanha peso e volume, e não o preço: um relógio de R$ 3.999 com 800 g paga R$ 17, enquanto itens de 20 a 30 kg pagam entre R$ 135 e R$ 410. Se houvesse erro de digitação no preço, essa relação não se sustentaria.

Seis linhas idênticas de um produto de sinalização (R$ 175, frete de R$ 299) não são duplicatas: pertencem ao mesmo pedido, com `order_item_id` de 1 a 6, ou seja, seis unidades. O frete do pedido é dividido entre os itens, e cada linha traz a sua parte.

In [0]:
SELECT
  CAST(i.price AS DECIMAL(10,2))              AS preco,
  CAST(i.freight_value AS DECIMAL(10,2))      AS frete,
  p.product_category_name                     AS categoria,
  CAST(p.product_weight_g AS INT)             AS peso_g,
  CAST(p.product_length_cm AS INT) * CAST(p.product_height_cm AS INT)
    * CAST(p.product_width_cm AS INT)         AS volume_cm3,
  i.order_id
FROM pospucrio.bronze.olist_order_items_dataset i
LEFT JOIN pospucrio.bronze.olist_products_dataset p USING (product_id)
ORDER BY CAST(i.price AS DECIMAL(10,2)) DESC
LIMIT 20

In [0]:
SELECT
  CAST(i.price AS DECIMAL(10,2))              AS preco,
  CAST(i.freight_value AS DECIMAL(10,2))      AS frete,
  p.product_category_name                     AS categoria,
  CAST(p.product_weight_g AS INT)             AS peso_g,
  CAST(p.product_length_cm AS INT) * CAST(p.product_height_cm AS INT)
    * CAST(p.product_width_cm AS INT)         AS volume_cm3,
  i.order_id
FROM pospucrio.bronze.olist_order_items_dataset i
LEFT JOIN pospucrio.bronze.olist_products_dataset p USING (product_id)
ORDER BY CAST(i.freight_value AS DECIMAL(10,2)) DESC
LIMIT 20

### 4.5 Outliers de preço: categorias acima do percentil 99

Os cerca de mil itens acima de R$ 889,99 são agrupados por categoria. Se a cauda se concentra em categorias onde preço alto é esperado, ela é real. A segunda consulta examina as categorias menos óbvias que aparecem nessa cauda.

**Resultado:** concentração em relógios, computadores, beleza e saúde (equipamentos de até 30 kg), esporte, informática, bebês, automotivo e eletroportáteis. Os casos isolados têm explicação plausível: box de DVDs de 2,6 kg vendido pelo mesmo preço em dois pedidos, livro de 13,3 kg (coleção), acessórios de moda de 100 g (joias ou óculos), item de papelaria de 21,8 kg (equipamento). Como o dataset não traz o nome dos produtos, não é possível confirmar cada caso, mas nenhum altera as respostas do projeto.

**Decisão:** outliers de preço e frete verificados e mantidos.

In [0]:
SELECT p.product_category_name AS categoria, COUNT(*) AS itens
FROM pospucrio.bronze.olist_order_items_dataset i
LEFT JOIN pospucrio.bronze.olist_products_dataset p USING (product_id)
WHERE CAST(i.price AS DECIMAL(10,2)) > 889.99
GROUP BY 1
ORDER BY itens DESC

In [0]:
SELECT
  p.product_category_name                 AS categoria,
  CAST(i.price AS DECIMAL(10,2))          AS preco,
  CAST(i.freight_value AS DECIMAL(10,2))  AS frete,
  CAST(p.product_weight_g AS INT)         AS peso_g,
  i.order_id
FROM pospucrio.bronze.olist_order_items_dataset i
LEFT JOIN pospucrio.bronze.olist_products_dataset p USING (product_id)
WHERE CAST(i.price AS DECIMAL(10,2)) > 889.99
  AND p.product_category_name IN ('papelaria', 'dvds_blu_ray', 'livros_interesse_geral',
                                  'musica', 'fashion_bolsas_e_acessorios')
ORDER BY categoria, preco DESC


### 4.6 Sequência do número do item

Em cada pedido, `order_item_id` deve ir de 1 até N sem lacunas. Um pedido cujo maior número difere da quantidade de itens indicaria registros faltando.

**Resultado:** 0 pedidos com lacuna.

In [0]:
SELECT COUNT(*) AS pedidos_com_lacuna
FROM (
  SELECT order_id
  FROM pospucrio.bronze.olist_order_items_dataset
  GROUP BY order_id
  HAVING MAX(CAST(order_item_id AS INT)) <> COUNT(*)
)

### 4.7 Integridade com pedidos, produtos e vendedores

Todo item deve apontar para um pedido, um produto e um vendedor existentes.

**Resultado:** 0, 0 e 0. Integridade completa.

In [0]:
SELECT
  (SELECT COUNT(*) FROM pospucrio.bronze.olist_order_items_dataset i
     LEFT ANTI JOIN pospucrio.bronze.olist_orders_dataset o USING (order_id))     AS sem_pedido,
  (SELECT COUNT(*) FROM pospucrio.bronze.olist_order_items_dataset i
     LEFT ANTI JOIN pospucrio.bronze.olist_products_dataset p USING (product_id)) AS sem_produto,
  (SELECT COUNT(*) FROM pospucrio.bronze.olist_order_items_dataset i
     LEFT ANTI JOIN pospucrio.bronze.olist_sellers_dataset s USING (seller_id))   AS sem_vendedor

### 4.8 Validação da prata

*Requer o notebook `Silver_Limpeza` executado.* A prata só aplica tipagem nesta tabela, então quantidade de linhas e soma dos preços devem ser idênticas nas duas camadas.

**Resultado:** 112.650 linhas e R$ 13.591.643,70 nas duas camadas. A conversão de tipos não perdeu nem alterou nenhum valor.

In [0]:
SELECT 'bronze' AS camada, COUNT(*) AS linhas, SUM(CAST(price AS DECIMAL(10,2))) AS total_preco
FROM pospucrio.bronze.olist_order_items_dataset
UNION ALL
SELECT 'silver', COUNT(*), SUM(price)
FROM pospucrio.silver.order_items

## 5. order_payments

Granularidade esperada: uma linha por pagamento, com chave composta `order_id` + `payment_sequential`. Um pedido pode ter vários pagamentos, de meios iguais ou diferentes.

As consultas desta seção sustentam três regras da camada ouro: como classificar um pedido em à vista ou parcelado, qual é o método principal de pagamento e se o voucher entra no valor pago.

### 5.1 Granularidade, tipos e valores

**Resultado:** 103.886 pagamentos e 103.886 chaves distintas, referentes a 99.440 pedidos. Nenhum valor inválido ou negativo, 9 pagamentos de valor zero, investigados em 5.4. Nenhuma parcela inválida.

In [0]:
SELECT
  COUNT(*)                                                       AS linhas,
  COUNT(DISTINCT order_id, payment_sequential)                   AS chaves_distintas,
  COUNT(DISTINCT order_id)                                       AS pedidos,
  COUNT_IF(try_cast(payment_value AS DECIMAL(10,2)) IS NULL)     AS valor_invalido,
  COUNT_IF(try_cast(payment_value AS DECIMAL(10,2)) < 0)         AS valor_negativo,
  COUNT_IF(try_cast(payment_value AS DECIMAL(10,2)) = 0)         AS valor_zero,
  COUNT_IF(try_cast(payment_installments AS INT) IS NULL)        AS parcela_invalida
FROM pospucrio.bronze.olist_order_payments_dataset

### 5.2 Tipo de pagamento por número de parcelas

Verifica se apenas o cartão de crédito tem mais de uma parcela, onde estão os registros com zero parcelas (inválidos, já que todo pagamento tem ao menos uma) e se há algum tipo inesperado.

**Resultado:** boleto, débito e voucher sempre com 1 parcela, cartão de crédito entre 1 e 24 parcelas. 2 pagamentos com cartão e zero parcelas. Um tipo inesperado: `not_defined`, com 3 registros.

In [0]:
SELECT payment_type, CAST(payment_installments AS INT) AS parcelas, COUNT(*) AS n
FROM pospucrio.bronze.olist_order_payments_dataset
GROUP BY 1, 2
ORDER BY 1, 2

### 5.3 Parcelas zeradas e status do pedido

Testa a hipótese de que os pagamentos com zero parcelas pertencem a pedidos cancelados.

**Resultado:** hipótese refutada. Os 2 pagamentos são de cartão de crédito em pedidos entregues. O cliente pagou e recebeu, apenas o número de parcelas não foi registrado.

**Decisão:** parcelas zeradas convertidas em nulo na prata. Na camada ouro, apenas a forma de pagamento desses pedidos fica nula, valor e demais dados permanecem.

In [0]:
SELECT o.order_status, p.payment_type, COUNT(*) AS n
FROM pospucrio.bronze.olist_order_payments_dataset p
JOIN pospucrio.bronze.olist_orders_dataset o USING (order_id)
WHERE CAST(p.payment_installments AS INT) = 0
GROUP BY 1, 2

### 5.4 Pagamentos de valor zero e tipo não definido

**Resultado:** os 9 pagamentos de valor zero se dividem em dois grupos:
- 3 `not_defined`, todos de pedidos cancelados: registros de pagamento vazios de pedidos que não se concretizaram.
- 6 vouchers em pedidos entregues ou enviados. Hipóteses: voucher aplicado sem saldo, ou artefato do sistema. Não é possível confirmar.

**Decisão:** mantidos sem alteração. Um voucher de valor zero não soma ao valor pago e, pela regra definida, o voucher não entra na classificação de parcelamento nem na escolha do método principal. Pedidos cancelados não entram nas perguntas de compra. Na camada ouro, o método principal dos pedidos `not_defined` fica nulo.

In [0]:
SELECT p.order_id, p.payment_type, p.payment_installments,
       p.payment_value, o.order_status
FROM pospucrio.bronze.olist_order_payments_dataset p
JOIN pospucrio.bronze.olist_orders_dataset o USING (order_id)
WHERE CAST(p.payment_value AS DECIMAL(10,2)) = 0
   OR p.payment_type = 'not_defined'
ORDER BY p.payment_type

### 5.5 Combinações de meios de pagamento por pedido

Junta os meios distintos usados em cada pedido. Mostra quantos pedidos combinam meios diferentes (o que exige uma regra para o método principal) e se existem pedidos pagos apenas com voucher.

**Resultado:** cartão de crédito (74.259 pedidos), boleto (19.784), cartão com voucher (2.245), **somente voucher (1.621)**, débito (1.527), `not_defined` (3) e cartão de crédito com débito (1).

**Decisões para a camada ouro:**
- O voucher nunca é o método principal quando há outro meio.
- Nos 1.621 pedidos pagos só com voucher, o método principal é o voucher.
- O único pedido com crédito e débito expõe a necessidade de desempate: prevalece o meio de maior valor pago.

In [0]:
SELECT tipos, COUNT(*) AS pedidos
FROM (
  SELECT order_id, array_join(array_sort(collect_set(payment_type)), ' + ') AS tipos
  FROM pospucrio.bronze.olist_order_payments_dataset
  GROUP BY order_id
)
GROUP BY tipos
ORDER BY pedidos DESC

### 5.6 Integridade e consistência com os itens

Verifica se todo pedido tem pagamento e se todo pagamento aponta para um pedido. Depois compara, pedido a pedido, o total pago com a soma de preço e frete dos itens (*requer a prata de itens*), com tolerância de um centavo.

**Resultado:** 98.362 pedidos batem, voucher incluído. Isso mostra que o voucher cobre parte do valor da compra, funcionando como meio de pagamento, e justifica incluí-lo no valor pago. 264 pedidos com pago maior e 39 com pago menor, investigados em 5.7.

Um pedido sem pagamento, de setembro de 2016: fora do recorte temporal, sem efeito sobre as análises.

In [0]:
WITH pago AS (
  SELECT order_id, SUM(CAST(payment_value AS DECIMAL(10,2))) AS total_pago
  FROM pospucrio.bronze.olist_order_payments_dataset
  GROUP BY order_id
),
itens AS (
  SELECT order_id, SUM(price + freight_value) AS total_itens
  FROM pospucrio.silver.order_items
  GROUP BY order_id
)
SELECT
  (SELECT COUNT(*) FROM pospucrio.bronze.olist_orders_dataset o
     LEFT ANTI JOIN pago USING (order_id))                              AS pedidos_sem_pagamento,
  (SELECT COUNT(*) FROM pago LEFT ANTI JOIN
     pospucrio.bronze.olist_orders_dataset o USING (order_id))          AS pagamentos_sem_pedido,
  COUNT_IF(abs(p.total_pago - i.total_itens) <= 0.01)                   AS batem,
  COUNT_IF(p.total_pago > i.total_itens + 0.01)                         AS pago_maior,
  COUNT_IF(p.total_pago < i.total_itens - 0.01)                         AS pago_menor
FROM pago p
JOIN itens i USING (order_id)

In [0]:
SELECT o.order_id, o.order_status, o.order_purchase_timestamp
FROM pospucrio.bronze.olist_orders_dataset o
LEFT ANTI JOIN pospucrio.bronze.olist_order_payments_dataset p USING (order_id)

### 5.7 Divergências entre valor pago e valor dos itens

Agrupa as divergências por meios de pagamento e por parcelamento, para testar a hipótese de que o valor pago maior se deve a juros.

**Resultado:**
- **Juros confirmados:** 246 dos 264 casos de pago maior são cartão parcelado, com R$ 12,40 a mais em média. Somando os 3 de cartão com voucher parcelados, são 249.
- **Arredondamento:** 14 boletos (8 a mais, 6 a menos) com diferença média de R$ 0,02. A tolerância de um centavo foi estreita demais, com cinco centavos, esses casos batem.
- **Sem causa identificável:** cerca de 40 pedidos com diferenças de R$ 1 a R$ 7, possivelmente descontos não registrados como voucher ou taxas.

**Consequência para a análise:** como o valor pago inclui juros, compras parceladas parecem ligeiramente mais caras que o valor dos produtos. Deve ser considerado na comparação entre compras à vista e parceladas.

In [0]:
WITH pago AS (
  SELECT order_id,
         SUM(CAST(payment_value AS DECIMAL(10,2)))              AS total_pago,
         MAX(CAST(payment_installments AS INT))                 AS max_parcelas,
         array_join(array_sort(collect_set(payment_type)), ' + ') AS tipos
  FROM pospucrio.bronze.olist_order_payments_dataset
  GROUP BY order_id
),
itens AS (
  SELECT order_id, SUM(price + freight_value) AS total_itens
  FROM pospucrio.silver.order_items
  GROUP BY order_id
),
comp AS (
  SELECT p.*, i.total_itens,
         p.total_pago - i.total_itens AS diferenca,
         CASE WHEN p.total_pago > i.total_itens + 0.01 THEN 'pago_maior'
              WHEN p.total_pago < i.total_itens - 0.01 THEN 'pago_menor' END AS situacao
  FROM pago p JOIN itens i USING (order_id)
)
SELECT situacao, tipos, max_parcelas > 1 AS parcelado,
       COUNT(*) AS pedidos, ROUND(AVG(diferenca), 2) AS diferenca_media
FROM comp
WHERE situacao IS NOT NULL
GROUP BY 1, 2, 3
ORDER BY 1, pedidos DESC

## 6. order_reviews

Tabela que contém os comentários com quebra de linha, tratados com a opção `multiLine` na ingestão, e as avaliações repetidas que a camada ouro precisa resolver.

### 6.1 Granularidade, nota e datas

A verificação da nota tem uma função adicional: se algum comentário com quebra de linha tivesse sido partido na ingestão, o texto cairia nas colunas seguintes e a nota apareceria como texto. As quatro contagens do topo revelam o formato das repetições.

**Resultado:**
- 99.224 linhas e 99.224 pares distintos de `review_id` e `order_id`: esta é a chave. Nenhuma das duas colunas é única sozinha.
- 98.410 `review_id` distintos: há avaliações associadas a mais de um pedido (investigado em 6.2).
- 98.673 `order_id` distintos: há pedidos com mais de uma avaliação (investigado em 6.3).
- Nenhuma nota fora de 1 a 5, confirmando que a ingestão preservou os comentários. Nenhuma data inválida nem resposta anterior à criação.
- 58.247 avaliações sem comentário (58,7%). Apenas 41% dos clientes escreveram algo, o que limita a representatividade de uma análise de texto.

In [0]:
SELECT
  COUNT(*)                                                         AS linhas,
  COUNT(DISTINCT review_id)                                        AS reviews_distintas,
  COUNT(DISTINCT order_id)                                         AS pedidos_distintos,
  COUNT(DISTINCT review_id, order_id)                              AS pares_distintos,
  COUNT_IF(try_cast(review_score AS INT) NOT BETWEEN 1 AND 5
        OR try_cast(review_score AS INT) IS NULL)                  AS nota_invalida,
  COUNT_IF(try_cast(review_creation_date AS TIMESTAMP) IS NULL)    AS criacao_invalida,
  COUNT_IF(try_cast(review_answer_timestamp AS TIMESTAMP) IS NULL) AS resposta_invalida,
  COUNT_IF(try_cast(review_answer_timestamp AS TIMESTAMP)
         < try_cast(review_creation_date AS TIMESTAMP))            AS resposta_antes_criacao,
  COUNT_IF(review_comment_message IS NULL)                         AS sem_comentario
FROM pospucrio.bronze.olist_order_reviews_dataset

### 6.2 O mesmo review_id em pedidos diferentes

A primeira consulta mostra exemplos, a segunda testa a explicação: se uma avaliação cobre uma compra dividida em vários pedidos, esses pedidos devem ser do mesmo cliente e da mesma data.

**Resultado:** as linhas de uma mesma avaliação são idênticas em nota, data e comentário, e os comentários mencionam compras de vários produtos ("o pedido consta de 2 produtos e até agora recebi apenas 1"). Das 789 avaliações repetidas, 789 pertencem a um único cliente e 748 (95%) à mesma data de compra. Os demais 41 casos, do mesmo cliente em datas diferentes, sugerem que a pesquisa está ligada à entrega: pedidos feitos em dias distintos que chegaram juntos.

**Decisão:** mantidas. O modelo atribui nota por pedido, e cada pedido recebeu de fato essa avaliação. Efeito colateral pequeno: na média por pedido, uma avaliação associada a dois pedidos pesa duas vezes (cerca de 800 casos em 99 mil).

In [0]:
SELECT r.review_id, r.order_id, r.review_score,
       r.review_creation_date, left(r.review_comment_message, 60) AS comentario
FROM pospucrio.bronze.olist_order_reviews_dataset r
JOIN (
  SELECT review_id
  FROM pospucrio.bronze.olist_order_reviews_dataset
  GROUP BY review_id
  HAVING COUNT(DISTINCT order_id) > 1
) d USING (review_id)
ORDER BY r.review_id
LIMIT 30


In [0]:
WITH repetidas AS (
  SELECT review_id
  FROM pospucrio.bronze.olist_order_reviews_dataset
  GROUP BY review_id
  HAVING COUNT(DISTINCT order_id) > 1
),
por_review AS (
  SELECT r.review_id,
         COUNT(DISTINCT c.customer_unique_id)                               AS clientes,
         COUNT(DISTINCT date(CAST(o.order_purchase_timestamp AS TIMESTAMP))) AS datas_compra
  FROM pospucrio.bronze.olist_order_reviews_dataset r
  JOIN repetidas USING (review_id)
  JOIN pospucrio.bronze.olist_orders_dataset o USING (order_id)
  JOIN pospucrio.silver.customers c USING (customer_id)
  GROUP BY r.review_id
)
SELECT COUNT(*)                   AS reviews_repetidas,
       COUNT_IF(clientes = 1)     AS mesmo_cliente,
       COUNT_IF(datas_compra = 1) AS mesma_data
FROM por_review


### 6.3 Pedidos com mais de uma avaliação

Mede duas coisas para a regra de escolha da avaliação: em quantos pedidos a escolha muda a nota, e em quantos a data de criação empata.

**Resultado:** 547 pedidos com mais de uma avaliação (543 com duas, 4 com três). Em 202 deles as notas diferem, então a regra altera o resultado. Em 157 há empate na data de criação, que só registra o dia.

**Decisão:** a data de criação é a de envio da pesquisa, o momento em que o cliente avaliou é a data de resposta, que tem hora completa. Prevalece a avaliação de resposta mais recente, em empate, a de criação mais recente, persistindo, o maior `review_id`, critério arbitrário usado apenas para garantir resultado determinístico.

In [0]:
WITH multi AS (
  SELECT order_id,
         COUNT(*)                              AS n_reviews,
         COUNT(DISTINCT review_score)          AS notas_diferentes,
         COUNT(DISTINCT review_creation_date)  AS datas_diferentes
  FROM pospucrio.bronze.olist_order_reviews_dataset
  GROUP BY order_id
  HAVING COUNT(*) > 1
)
SELECT n_reviews,
       COUNT(*)                                AS pedidos,
       COUNT_IF(notas_diferentes > 1)          AS com_notas_diferentes,
       COUNT_IF(datas_diferentes < n_reviews)  AS com_empate_de_data
FROM multi
GROUP BY n_reviews
ORDER BY n_reviews

### 6.4 Integridade com pedidos

**Resultado:** nenhuma avaliação sem pedido. 768 pedidos sem avaliação (menos de 1%): não é erro, nem todo cliente avalia. Esses pedidos ficam com nota nula na camada ouro e fora das médias.

In [0]:
SELECT
  (SELECT COUNT(*) FROM pospucrio.bronze.olist_order_reviews_dataset r
     LEFT ANTI JOIN pospucrio.bronze.olist_orders_dataset o USING (order_id)) AS reviews_sem_pedido,
  (SELECT COUNT(*) FROM pospucrio.bronze.olist_orders_dataset o
     LEFT ANTI JOIN pospucrio.bronze.olist_order_reviews_dataset r USING (order_id)) AS pedidos_sem_review

### 6.5 Distribuição das notas

Não é uma verificação de erro, mas contexto necessário para a análise.

**Resultado:** nota 5 (57,8%), 4 (19,3%), 3 (8,2%), 2 (3,2%), 1 (11,5%). Média geral de aproximadamente 4,09. A distribuição é polarizada: a nota 1 é mais frequente que a 2 e a 3 somadas. O cliente insatisfeito tende a ir direto ao extremo.

**Consequência para a análise:** a média esconde essa polarização. Nas comparações entre estados e trimestres, a proporção de notas 1 e 2 deve ser reportada ao lado da média.

In [0]:
SELECT CAST(review_score AS INT) AS nota, COUNT(*) AS n,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM pospucrio.bronze.olist_order_reviews_dataset
GROUP BY 1
ORDER BY 1

## 7. products

Granularidade esperada: uma linha por produto. Os nomes `product_name_lenght` e `product_description_lenght` estão grafados errado na origem, as consultas sobre o bronze usam a grafia original.

### 7.1 Granularidade, nulos e tipos

**Resultado:** 32.951 produtos, sem duplicatas. 610 sem categoria, nome, descrição e fotos, 2 sem peso e dimensões. Nenhum peso inválido, 4 produtos com peso zero, valor impossível.

**Decisão:** peso zero convertido em nulo na prata.

In [0]:
SELECT
  COUNT(*)                                                    AS linhas,
  COUNT(DISTINCT product_id)                                  AS produtos_distintos,
  COUNT_IF(product_category_name IS NULL)                     AS sem_categoria,
  COUNT_IF(product_name_lenght IS NULL)                       AS sem_tam_nome,
  COUNT_IF(product_description_lenght IS NULL)                AS sem_tam_descricao,
  COUNT_IF(product_photos_qty IS NULL)                        AS sem_fotos,
  COUNT_IF(product_weight_g IS NULL)                          AS sem_peso,
  COUNT_IF(product_length_cm IS NULL OR product_height_cm IS NULL
        OR product_width_cm IS NULL)                          AS sem_dimensao,
  COUNT_IF(product_weight_g IS NOT NULL
       AND try_cast(product_weight_g AS INT) IS NULL)         AS peso_invalido,
  COUNT_IF(try_cast(product_weight_g AS INT) = 0)             AS peso_zero
FROM pospucrio.bronze.olist_products_dataset

### 7.2 Os nulos aparecem juntos?

Se os produtos sem categoria são os mesmos sem nome, descrição e fotos, o problema é de cadastro incompleto, e não de falhas isoladas em colunas.

**Resultado:** 609 produtos sem categoria, nome e fotos, mas com peso, 1 sem nada, 1 com tudo exceto peso. Os 610 sem categoria são cadastros incompletos.

In [0]:
SELECT
  product_category_name IS NULL                   AS sem_categoria,
  product_name_lenght IS NULL                     AS sem_tam_nome,
  product_photos_qty IS NULL                      AS sem_fotos,
  product_weight_g IS NULL                        AS sem_peso,
  COUNT(*)                                        AS produtos
FROM pospucrio.bronze.olist_products_dataset
GROUP BY 1, 2, 3, 4
ORDER BY produtos DESC

### 7.3 Peso e dimensões

A primeira consulta lista os pesos mais frequentes, para testar a suspeita, levantada em 4.4, de que valores redondos (30.000 g, 100 g) seriam preenchimento padrão do sistema. A segunda mede a distribuição e os máximos.

**Resultado:** os pesos mais frequentes são todos redondos (200, 300, 150, 400 g), mas se distribuem de forma gradual, típica de arredondamento no cadastro. 30.000 g não está entre os mais frequentes, e 100 g aparece com frequência semelhante à dos vizinhos. Suspeita não confirmada.

In [0]:
SELECT CAST(product_weight_g AS INT) AS peso_g, COUNT(*) AS produtos
FROM pospucrio.bronze.olist_products_dataset
GROUP BY 1
ORDER BY produtos DESC
LIMIT 15

In [0]:
SELECT
  percentile_approx(CAST(product_weight_g AS INT), array(0.01, 0.5, 0.99)) AS peso_percentis,
  MIN(CAST(product_weight_g AS INT))                                       AS peso_min,
  MAX(CAST(product_weight_g AS INT))                                       AS peso_max,
  MAX(CAST(product_length_cm AS INT))                                      AS comp_max,
  MAX(CAST(product_height_cm AS INT))                                      AS alt_max,
  MAX(CAST(product_width_cm AS INT))                                       AS larg_max
FROM pospucrio.bronze.olist_products_dataset

### 7.4 Peso dos produtos sem categoria nas vendas

O que importa para a pergunta sobre categorias é quantos itens vendidos não têm categoria, e não quantos produtos.

**Resultado:** 1.603 itens (1,42% das vendas). Não altera a categoria de maior demanda, já que as maiores têm vários milhares de itens.

**Decisão:** agrupados como "sem categoria" na camada ouro, para que a soma das categorias continue batendo com o total de itens vendidos.

In [0]:
SELECT COUNT(*) AS itens_sem_categoria,
       ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM pospucrio.silver.order_items), 2) AS pct
FROM pospucrio.silver.order_items i
JOIN pospucrio.bronze.olist_products_dataset p USING (product_id)
WHERE p.product_category_name IS NULL

### 7.5 Valores de categoria

Com cerca de 70 valores, o domínio é pequeno o bastante para leitura completa. Em ordem alfabética, variantes de um mesmo nome aparecem lado a lado.

**Resultado:** formato consistente (minúsculas, sem acento, com sublinhado) e nenhuma grafia variante. Há sobreposições de significado (`alimentos`, `alimentos_bebidas` e `bebidas`, `pcs`, `pc_gamer` e `informatica_acessorios`), que refletem a granularidade escolhida pela origem e são mantidas. Categorias genéricas (`cool_stuff`, `market_place`) e uma de serviços (`seguros_e_servicos`) indicam que nem todo item do catálogo é produto físico descrito. Dois pares com sufixo `_2` investigados em 7.6.

In [0]:
SELECT product_category_name, COUNT(*) AS produtos
FROM pospucrio.silver.products
GROUP BY 1
ORDER BY 1

### 7.6 Categorias com sufixo _2

Se as versões `_2` tiverem perfil de preço e peso diferente das homônimas, são categorias distintas, se forem iguais, podem ser duplicação.

**Resultado:** perfis claramente diferentes. `eletrodomesticos_2` tem preço mediano de R$ 223,90 e 7,5 kg, contra R$ 47,59 e 475 g de `eletrodomesticos`. `casa_conforto_2` tem R$ 12,90 e 700 g, contra R$ 89,90 e 1,8 kg.

**Decisão:** categorias mantidas separadas e com os nomes originais. Renomear (por exemplo, "eletrodomésticos grandes") seria afirmar uma inferência que a fonte não faz.

In [0]:
SELECT p.product_category_name,
       COUNT(*)                                     AS itens,
       percentile_approx(i.price, 0.5)              AS preco_mediano,
       percentile_approx(p.product_weight_g, 0.5)   AS peso_mediano_g
FROM pospucrio.silver.order_items i
JOIN pospucrio.silver.products p USING (product_id)
WHERE p.product_category_name IN ('eletrodomesticos', 'eletrodomesticos_2',
                                  'casa_conforto', 'casa_conforto_2')
GROUP BY 1
ORDER BY 1

### 7.7 Cobertura da tabela de tradução

Lista as categorias sem correspondência na tabela de tradução para o inglês.

**Resultado:** 2 categorias sem tradução (`pc_gamer` e `portateis_cozinha_e_preparadores_de_alimentos`, 13 produtos).

**Decisão:** a análise é feita em português, com os nomes originais das categorias. A tabela de tradução permanece no bronze, sem uso.

In [0]:
SELECT
  p.product_category_name,
  COUNT(*) AS produtos
FROM pospucrio.bronze.olist_products_dataset p
LEFT ANTI JOIN pospucrio.bronze.product_category_name_translation t
  USING (product_category_name)
WHERE p.product_category_name IS NOT NULL
GROUP BY 1